# Robustness check: final definition vs. broad wildcard variant

**Purpose**: Test whether the **time trends of relative distributions** remain stable
when using a broader wildcard variant instead of the final, strict definition.

**Period**: 2022-04-21 to 2025-02-08 (scraper cutoff to last crawled date; update 2026-03-20)

**Main definition (final, as in notebook 03)**:
- Klimawandel: wandel, wandels
- Klimakrise: krise, krisen
- Klimaschutz: schutz, schutzes

**Comparison variant (broad)**:
- Klimawandel: wandel*
- Klimakrise: krise*
- Klimaschutz: schutz*

## Setup: imports and data loading

In [ ]:
import os
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# phinx: Sphinx documentation tool.

notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))

from analysis_utils import map_suffix_by_prefix, map_suffix_to_term
from config import DEFAULT_END_DATE, DWH_DB_PATH, PLOTS_DIR, SCRAPER_CUTOFF
from handle_sqlite import read_table_as_dataframe
from plotting_utils import apply_plot_style

apply_plot_style()

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
plot_dir = PLOTS_DIR

# Load processed tables from SQLite
df_context = read_table_as_dataframe("context_processed", str(DWH_DB_PATH))
df_meta = read_table_as_dataframe("newspapers", str(DWH_DB_PATH))

print(f"Context rows: {len(df_context)}")
print(f"Metadata rows: {len(df_meta)}")
print(f"Context columns: {df_context.columns.tolist()}")

In [ ]:
df_meta['data_published'] = pd.to_datetime(df_meta['data_published'])

We keep a left join on metadata: the ~17,004 left-join rows are metadata entries without a matching context row.
That can mean there was no "Klima" hit for that newspaper/day, or the context extraction produced no row.
Because we want to keep the option to show newspaper-days without Klima mentions, we retain them.
We also have more context entries because we apply a date cutoff, so rows outside the window are dropped by the join.


In [ ]:
analysis_start = SCRAPER_CUTOFF
analysis_end = DEFAULT_END_DATE

df_meta_filtered = df_meta[df_meta["data_published"] >= analysis_start].copy()
if analysis_end is not None:
    df_meta_filtered = df_meta_filtered[df_meta_filtered["data_published"] <= analysis_end].copy()

analysis_end_label = analysis_end.date() if analysis_end is not None else "open end"
print(f"Total metadata rows: {len(df_meta)}")
print(f"Metadata rows from {analysis_start.date()} to {analysis_end_label}: {len(df_meta_filtered)}")

In [ ]:
# Check for null values in key columns before inner join
print("=== NULL VALUES IN df_context ===")
print(df_context.isnull().sum())
print(f"\nTotal rows in df_context: {len(df_context)}")

print("\n=== NULL VALUES IN df_meta_filtered ===")
print(df_meta_filtered.isnull().sum())
print(f"\nTotal rows in df_meta_filtered: {len(df_meta_filtered)}")

# Show what will be lost in inner join
print("\n=== IMPACT OF INNER JOIN ===")
print(f"Rows in df_meta_filtered: {len(df_meta_filtered)}")
print(f"Rows in df_context (before filter): {len(df_context)}")

# Check how many context rows have matching newspaper_ids in df_meta_filtered
matching_ids = df_context['newspaper_id'].isin(df_meta_filtered['newspaper_id']).sum()
print(f"Context rows with matching newspaper_id in df_meta_filtered: {matching_ids}")
print(f"Context rows that will be lost: {len(df_context) - matching_ids}")

In [ ]:
# Merge: meta is left → all meta rows kept, context joined where available
df = df_meta_filtered[['newspaper_id', 'newspaper_name', 'data_published']].merge(df_context, on='newspaper_id', how='left')

print(f"Totale Einträge von {SCRAPER_CUTOFF} bis {END_DATE}: {len(df)} Nennungen")

In [ ]:
print(f"Zeitungen: {df['newspaper_name'].nunique()}")
print(f"\nSpalten in df: {df.columns.tolist()}")

## Comparison: relative overall distributions

In [ ]:
# Final mapping definition after processing
SUFFIX_TO_TERM = {
    "wandel": "Klimawandel",
    "krise": "Klimakrise",
    "schutz": "Klimaschutz",
}

df["term"] = (
    df["suffix_lemma"]
    .str.lower()
    .str.strip()
    .map(SUFFIX_TO_TERM)
)

print("\nFinal suffixes used:")
for suffix, term in SUFFIX_TO_TERM.items():
    print(f"- {term}: {suffix}")

df_three_terms = df[df["term"].notna()].copy()
print(f"Mentions of the three terms (final selection): {len(df_three_terms)}")
print(df_three_terms["term"].value_counts())

## Quarterly comparison: relative shares and trend similarity

In [ ]:
# Main definition: exact final suffix list
MAIN_SUFFIX_MAP = {
    "wandel": "Klimawandel",
    "wandels": "Klimawandel",
    "krise": "Klimakrise",
    "krisen": "Klimakrise",
    "schutz": "Klimaschutz",
    "schutzes": "Klimaschutz",
}

df["suffix_clean"] = df["suffix"].str.lower().str.strip()
df_main = df.copy()
df_main["term"] = df_main["suffix_clean"].apply(
    lambda value: map_suffix_to_term(value, MAIN_SUFFIX_MAP)
 )
df_main = df_main[df_main["term"].notna()].copy()

print("\n=== Main definition (final, notebook 03) ===")
print(f"Mentions: {len(df_main)}")
print("\nDistribution:")
print(df_main["term"].value_counts())
print("\nFinal suffix list:")
for suffix, term in MAIN_SUFFIX_MAP.items():
    print(f"- {term}: {suffix}")

In [ ]:
# Comparison variant: broad wildcard suffixes
PREFIX_MAP = {
    "wandel": "Klimawandel",
    "krise": "Klimakrise",
    "schutz": "Klimaschutz",
}

df_wildcard = df.copy()
df_wildcard["term"] = df_wildcard["suffix_clean"].apply(
    lambda value: map_suffix_by_prefix(value, PREFIX_MAP)
 )
df_wildcard = df_wildcard[df_wildcard["term"].notna()].copy()

print("\n=== Comparison variant (broad: wandel*, krise*, schutz*) ===")
print(f"Mentions: {len(df_wildcard)}")
print("\nDistribution:")
print(df_wildcard["term"].value_counts())
print("\nMost frequent suffix variants (top 15):")
print(df_wildcard["suffix_clean"].value_counts().head(15))

## Figure: deviations with wildcard vs. main definition

In [ ]:
# Compare relative overall distributions (not absolute levels)
print("\n=== Relative overall distribution: main definition vs. wildcard ===")
print()

terms = ["Klimawandel", "Klimakrise", "Klimaschutz"]
main_counts = df_main["term"].value_counts().reindex(terms, fill_value=0)
wild_counts = df_wildcard["term"].value_counts().reindex(terms, fill_value=0)

main_shares_total = (main_counts / main_counts.sum() * 100).round(2)
wild_shares_total = (wild_counts / wild_counts.sum() * 100).round(2)

comparison = pd.DataFrame({
    "Main share (%)": main_shares_total,
    "Wildcard share (%)": wild_shares_total,
})
comparison["Diff (pp)"] = (comparison["Wildcard share (%)"] - comparison["Main share (%)"]).round(2)

print(comparison.to_string())
print()
print(f"Main definition mentions: {int(main_counts.sum())}")
print(f"Wildcard mentions: {int(wild_counts.sum())}")
print("Note: Relative shares are the focus of this robustness check.")

## Quarterly comparison: relative shares and trend similarity

In [ ]:
# Quarterly aggregation
main_copy = df_main.copy()
main_copy["quarter"] = main_copy["data_published"].dt.to_period("Q")
main_quarterly = main_copy.groupby(["quarter", "term"]).size().unstack(fill_value=0)

wild_copy = df_wildcard.copy()
wild_copy["quarter"] = wild_copy["data_published"].dt.to_period("Q")
wild_quarterly = wild_copy.groupby(["quarter", "term"]).size().unstack(fill_value=0)

# Align quarters/terms for fair comparison
quarter_index = main_quarterly.index.union(wild_quarterly.index)
term_index = ["Klimawandel", "Klimakrise", "Klimaschutz"]
main_quarterly = main_quarterly.reindex(index=quarter_index, columns=term_index, fill_value=0)
wild_quarterly = wild_quarterly.reindex(index=quarter_index, columns=term_index, fill_value=0)

# Relative shares per quarter
main_shares = (main_quarterly.div(main_quarterly.sum(axis=1), axis=0) * 100).fillna(0)
wild_shares = (wild_quarterly.div(wild_quarterly.sum(axis=1), axis=0) * 100).fillna(0)

print("\n=== Relative quarterly shares: main definition ===")
print(main_shares.round(2).to_string())
print("\n=== Relative quarterly shares: wildcard ===")
print(wild_shares.round(2).to_string())

# Trend similarity
share_diff = (wild_shares - main_shares).round(2)
mean_abs_diff = share_diff.abs().mean().round(2)
trend_corr = main_shares.corrwith(wild_shares).round(3)

print("\n=== Mean absolute deviation (pp) per term ===")
print(mean_abs_diff.to_string())
print("\n=== Correlation of quarterly share trends ===")
print(trend_corr.to_string())

## Figure: deviations with wildcard vs. main definition

In [ ]:
# Single-plot view with deviation segments
terms_plot = ["Klimaschutz", "Klimawandel", "Klimakrise"]
linestyles = {"Klimaschutz": "-", "Klimawandel": "--", "Klimakrise": ":"}
markers = {"Klimaschutz": "o", "Klimawandel": "s", "Klimakrise": "^"}

plot_index = main_shares.index
x = np.arange(len(plot_index))

fig, ax = plt.subplots(figsize=(12, 6))

for term in terms_plot:
    base = main_shares[term].reindex(plot_index).values
    wild = wild_shares[term].reindex(plot_index).values

    # Main definition
    ax.plot(
        x,
        base,
        linestyle=linestyles[term],
        marker=markers[term],
        color="black",
        linewidth=1.8,
        markersize=5,
        label=f"{term} (main)",
)

    # Wildcard variant
    ax.plot(
        x,
        wild,
        linestyle=linestyles[term],
        marker="x",
        color="dimgray",
        linewidth=1.0,
        markersize=5,
        alpha=0.8,
        label=f"{term} (wildcard)",
)

    # Vertical segments show deviations
    for xi, y0, y1 in zip(x, base, wild):
        ax.plot([xi, xi], [y0, y1], color="0.75", linewidth=0.8, alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in plot_index], rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("Relative share (%)")
ax.set_title("Deviations with wildcard vs. main definition")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(loc="best", ncol=2, frameon=True)
ax.text(
    0.01,
    -0.2,
    "Gray segments = deviation between main and wildcard per quarter",
    transform=ax.transAxes,
    fontsize=9,
)

plt.tight_layout()
plot_path = os.path.join(plot_dir, "06_abweichungen_lemma_statt_begriffsauswahl.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figure saved: {plot_path}")
plt.show()

## Additional checks: extra variants and trend differences

In [ ]:
print("\n" + "=" * 80)
print("ADDITIONAL CHECK: wildcard extras and trend differences")
print("=" * 80)

# Which suffixes appear only in the wildcard variant?
main_suffixes = set(df_main["suffix_clean"].dropna().unique())
wild_suffixes = set(df_wildcard["suffix_clean"].dropna().unique())
new_in_wild = sorted(wild_suffixes - main_suffixes)

print(f"\nUnique suffixes (main): {len(main_suffixes)}")
print(f"Unique suffixes (wildcard): {len(wild_suffixes)}")
print(f"Extra suffixes in wildcard: {len(new_in_wild)}")

if new_in_wild:
    extra_counts = (
        df_wildcard[df_wildcard["suffix_clean"].isin(new_in_wild)]["suffix_clean"]
        .value_counts()
        .head(20)
)
    print("\nTop-20 extra suffixes (wildcard only):")
    print(extra_counts.to_string())
else:
    print("No extra suffixes found.")

print("\nMean absolute quarterly deviation (pp) per term:")
print(mean_abs_diff.to_string())
print("\nCorrelation of quarterly shares per term:")
print(trend_corr.to_string())
print("\n" + "=" * 80)

## Conclusion

In [ ]:
print("\n" + "=" * 72)
print("CONCLUSION: main definition vs. wildcard variant")
print("=" * 72)

total_main = len(df_main)
total_wild = len(df_wildcard)
difference = total_wild - total_main
pct_increase = (difference / total_main * 100) if total_main > 0 else 0

print("\n1) DEFINITIONS:")
print("   - Main (final): wandel, wandels, krise, krisen, schutz, schutzes")
print("   - Wildcard: wandel*, krise*, schutz*")

print("\n2) COVERAGE (context-only):")
print(f"   - Main: {total_main:,} mentions")
print(f"   - Wildcard: {total_wild:,} mentions")
print(f"   - Difference: {difference:,} ({pct_increase:.1f}%)")

print("\n3) RELATIVE OVERALL DISTRIBUTION (diff in percentage points):")
for term in ["Klimawandel", "Klimakrise", "Klimaschutz"]:
    diff = comparison.loc[term, "Diff (pp)"]
    print(f"   - {term}: {diff:+.2f} pp")

print("\n4) QUARTERLY TREND SIMILARITY:")
for term in ["Klimawandel", "Klimakrise", "Klimaschutz"]:
    mad = mean_abs_diff.get(term, float("nan"))
    corr = trend_corr.get(term, float("nan"))
    print(f"   - {term}: mean deviation {mad:.2f} pp, correlation {corr:.3f}")

if (trend_corr.fillna(0) >= 0.9).all():
    print("\nAssessment: Relative trends remain very similar (robust).")
elif (trend_corr.fillna(0) >= 0.8).all():
    print("\nAssessment: Relative trends remain largely similar (robust).")
else:
    print("\nAssessment: Noticeable trend deviations between variants.")

print("\n5) DATA QUALITY:")
print(f"   - Time filter: {analysis_start.date()} to {analysis_end_label}")
print(f"   - Newspapers in sample: {df['newspaper_name'].nunique()}")
print("   - Scraper change on 2022-04-21 accounted for")

print("\n" + "=" * 72)